In [0]:
%sql
-- Set catalog
USE CATALOG labour_market_platform;
-- Create schema if not exists
CREATE SCHEMA IF NOT EXISTS silver_jobtech;
-- Use schema
USE SCHEMA silver_jobtech;

In [0]:
%sql

SELECT *
FROM bronze_jobtech.current_ads_raw
LIMIT 10;

In [0]:
%sql

DESCRIBE bronze_jobtech.current_ads_raw;

From the inspections above we can already confirm that:
- current_ads_raw is already flattened at the top level
- but it still contains several **nested structs / arrays** such as:
    - employer
    - occupation
    - occupation_field
    - occupation_group
    - salary_type
    - scope_of_work
    - working_hours_type
    - workplace_address
    - must_have
    - nice_to_have
That means Silver should be the layer where we:
- extract the business-useful scalar fields
- standardize names
- align current + historical into one canonical job-posting shape

But before we do that, we need to confirm whether  **historical** has the same structure.

In [0]:
%sql
DESCRIBE bronze_jobtech.historical_ads_raw

## Common core between current and historical

Both tables share the main job-posting fields we need for Silver, such as:

- id
- headline
- description
- application_deadline
- duration
- employer
- employment_type
- occupation
- occupation_field
- occupation_group
- number_of_vacancies
- publication_date
- last_publication_date
- salary_type
- scope_of_work
- working_hours_type
- workplace_address
- webpage_url
- removed
- removed_date

## Schema differences

Historical has some extra fields like:

- detected_language
- keywords
- remote_work
- open_for_all
- trainee
- franchise
- hire_work_place

Current has one field not in historical:

- relevance

That means the right Silver strategy is:

- build one canonical Silver table
- keep the shared business fields
- flatten the important nested structs
- add a record_source column so we know whether the row came from historical or current

In [0]:
%sql
SELECT
    id AS job_id,
    headline AS job_title,
    description,
    employer.name AS employer_name,
    occupation[0].label AS occupation_label,
    occupation_field[0].label AS occupation_field_label,
    occupation_group[0].label AS occupation_group_label,
    workplace_address.municipality_code AS municipality_code,
    workplace_address.city AS city,
    employment_type[0].label AS employment_type_label,
    duration.label AS duration_label,
    scope_of_work.min AS scope_of_work_min,
    scope_of_work.max AS scope_of_work_max,
    working_hours_type.label AS working_hours_type_label,
    salary_type.label AS salary_type_label,
    number_of_vacancies,
    application_deadline,
    publication_date,
    last_publication_date,
    removed,
    removed_date,
    webpage_url,
    'historical' AS record_source
FROM bronze_jobtech.historical_ads_raw
LIMIT 10;

In [0]:
%sql
-- Preview unified Silver JobTech dataset
SELECT
    id AS job_id,
    headline AS job_title,
    description,
    employer.name AS employer_name,
    element_at(occupation, 1).label AS occupation_label,
    element_at(occupation_field, 1).label AS occupation_field_label,
    element_at(occupation_group, 1).label AS occupation_group_label,
    workplace_address.municipality_code AS municipality_code,
    workplace_address.city AS city,
    element_at(employment_type, 1).label AS employment_type_label,
    duration.label AS duration_label,
    scope_of_work.min AS scope_of_work_min,
    scope_of_work.max AS scope_of_work_max,
    working_hours_type.label AS working_hours_type_label,
    salary_type.label AS salary_type_label,
    number_of_vacancies,
    application_deadline,
    publication_date,
    last_publication_date,
    removed,
    removed_date,
    webpage_url,
    'historical' AS record_source
FROM bronze_jobtech.historical_ads_raw

UNION ALL

SELECT
    id AS job_id,
    headline AS job_title,
    description,
    employer.name AS employer_name,
    occupation.label AS occupation_label,
    occupation_field.label AS occupation_field_label,
    occupation_group.label AS occupation_group_label,
    workplace_address.municipality_code AS municipality_code,
    workplace_address.city AS city,
    employment_type.label AS employment_type_label,
    duration.label AS duration_label,
    scope_of_work.min AS scope_of_work_min,
    scope_of_work.max AS scope_of_work_max,
    working_hours_type.label AS working_hours_type_label,
    salary_type.label AS salary_type_label,
    number_of_vacancies,
    application_deadline,
    publication_date,
    last_publication_date,
    removed,
    removed_date,
    webpage_url,
    'current' AS record_source
FROM bronze_jobtech.current_ads_raw
LIMIT 20;

In [0]:
%sql
-- Create unified Silver JobTech table

CREATE OR REPLACE TABLE silver_jobtech.job_postings_silver AS

SELECT
    id AS job_id,
    headline AS job_title,
    description,
    employer.name AS employer_name,
    element_at(occupation, 1).label AS occupation_label,
    element_at(occupation_field, 1).label AS occupation_field_label,
    element_at(occupation_group, 1).label AS occupation_group_label,
    workplace_address.municipality_code AS municipality_code,
    workplace_address.city AS city,
    element_at(employment_type, 1).label AS employment_type_label,
    duration.label AS duration_label,
    scope_of_work.min AS scope_of_work_min,
    scope_of_work.max AS scope_of_work_max,
    working_hours_type.label AS working_hours_type_label,
    salary_type.label AS salary_type_label,
    number_of_vacancies,
    application_deadline,
    publication_date,
    last_publication_date,
    removed,
    removed_date,
    webpage_url,
    'historical' AS record_source
FROM bronze_jobtech.historical_ads_raw

UNION ALL

SELECT
    id AS job_id,
    headline AS job_title,
    description,
    employer.name AS employer_name,
    occupation.label AS occupation_label,
    occupation_field.label AS occupation_field_label,
    occupation_group.label AS occupation_group_label,
    workplace_address.municipality_code AS municipality_code,
    workplace_address.city AS city,
    employment_type.label AS employment_type_label,
    duration.label AS duration_label,
    scope_of_work.min AS scope_of_work_min,
    scope_of_work.max AS scope_of_work_max,
    working_hours_type.label AS working_hours_type_label,
    salary_type.label AS salary_type_label,
    number_of_vacancies,
    application_deadline,
    publication_date,
    last_publication_date,
    removed,
    removed_date,
    webpage_url,
    'current' AS record_source
FROM bronze_jobtech.current_ads_raw;

In [0]:
%sql
SELECT *
FROM silver_jobtech.job_postings_silver
LIMIT 20;

In [0]:
%sql
SELECT
    record_source,
    COUNT(*) AS row_count
FROM silver_jobtech.job_postings_silver
GROUP BY record_source;

## Data quality check

In [0]:
%sql

SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN job_id IS NULL THEN 1 ELSE 0 END) AS null_job_id,
    SUM(CASE WHEN job_title IS NULL THEN 1 ELSE 0 END) AS null_job_title,
    SUM(CASE WHEN employer_name IS NULL THEN 1 ELSE 0 END) AS null_employer,
    SUM(CASE WHEN publication_date IS NULL THEN 1 ELSE 0 END) AS null_publication_date
FROM silver_jobtech.job_postings_silver;

In [0]:
%sql
-- Data Quality: Check duplicate job_ids
SELECT
    job_id,
    COUNT(*) AS occurrences
FROM silver_jobtech.job_postings_silver
GROUP BY job_id
HAVING COUNT(*) > 1
ORDER BY occurrences DESC
LIMIT 20;

The result of duplicate check shows multiple rows per job_id likely from:
- historical + current overlap
- updates of the same ad
to deduplicate I am going to keep the most recent version of each job.

In [0]:
%sql
-- Groups rows by job_id
-- Sorts each group by newest publication_date
-- Keep only the latest version rn = 1
SELECT *
FROM (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY job_id
               ORDER BY publication_date DESC
           ) AS rn
    FROM silver_jobtech.job_postings_silver
)
WHERE rn = 1
LIMIT 20;

In [0]:
%sql
-- Create deduplicated Silver table

CREATE OR REPLACE TABLE silver_jobtech.job_postings_silver_dedup AS

SELECT
    job_id,
    job_title,
    description,
    employer_name,
    occupation_label,
    occupation_field_label,
    occupation_group_label,
    municipality_code,
    city,
    employment_type_label,
    duration_label,
    scope_of_work_min,
    scope_of_work_max,
    working_hours_type_label,
    salary_type_label,
    number_of_vacancies,
    application_deadline,
    publication_date,
    last_publication_date,
    removed,
    removed_date,
    webpage_url,
    record_source
FROM (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY job_id
               ORDER BY publication_date DESC
           ) AS rn
    FROM silver_jobtech.job_postings_silver
)
WHERE rn = 1;

In [0]:
%sql
SELECT
    job_id,
    COUNT(*) AS occurrences
FROM silver_jobtech.job_postings_silver_dedup
GROUP BY job_id
HAVING COUNT(*) > 1
ORDER BY occurrences DESC
LIMIT 20;

In [0]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT job_id) AS distinct_job_ids
FROM silver_jobtech.job_postings_silver_dedup;

total_rows = distinct_job_ids

Deduplication is fully successful and persisted.

So now the Silver base table is clean at the primary key level (job_id)

In [0]:
%sql
DESCRIBE silver_jobtech.job_postings_silver_dedup;

In [0]:
%sql
SELECT
    COUNT(*) AS total_rows,

    SUM(CASE WHEN job_id IS NULL THEN 1 ELSE 0 END) AS null_job_id,
    SUM(CASE WHEN job_title IS NULL THEN 1 ELSE 0 END) AS null_job_title,
    SUM(CASE WHEN employer_name IS NULL THEN 1 ELSE 0 END) AS null_employer_name,

    SUM(CASE WHEN occupation_label IS NULL THEN 1 ELSE 0 END) AS null_occupation_label,
    SUM(CASE WHEN occupation_field_label IS NULL THEN 1 ELSE 0 END) AS null_occupation_field_label,
    SUM(CASE WHEN occupation_group_label IS NULL THEN 1 ELSE 0 END) AS null_occupation_group_label,

    SUM(CASE WHEN municipality_code IS NULL THEN 1 ELSE 0 END) AS null_municipality_code,
    SUM(CASE WHEN city IS NULL THEN 1 ELSE 0 END) AS null_city,

    SUM(CASE WHEN employment_type_label IS NULL THEN 1 ELSE 0 END) AS null_employment_type_label,
    SUM(CASE WHEN duration_label IS NULL THEN 1 ELSE 0 END) AS null_duration_label,
    SUM(CASE WHEN working_hours_type_label IS NULL THEN 1 ELSE 0 END) AS null_working_hours_type_label,
    SUM(CASE WHEN salary_type_label IS NULL THEN 1 ELSE 0 END) AS null_salary_type_label,

    SUM(CASE WHEN scope_of_work_min IS NULL THEN 1 ELSE 0 END) AS null_scope_of_work_min,
    SUM(CASE WHEN scope_of_work_max IS NULL THEN 1 ELSE 0 END) AS null_scope_of_work_max,
    SUM(CASE WHEN number_of_vacancies IS NULL THEN 1 ELSE 0 END) AS null_number_of_vacancies,

    SUM(CASE WHEN application_deadline IS NULL THEN 1 ELSE 0 END) AS null_application_deadline,
    SUM(CASE WHEN publication_date IS NULL THEN 1 ELSE 0 END) AS null_publication_date,
    SUM(CASE WHEN last_publication_date IS NULL THEN 1 ELSE 0 END) AS null_last_publication_date,
    SUM(CASE WHEN removed_date IS NULL THEN 1 ELSE 0 END) AS null_removed_date,

    SUM(CASE WHEN webpage_url IS NULL THEN 1 ELSE 0 END) AS null_webpage_url,
    SUM(CASE WHEN record_source IS NULL THEN 1 ELSE 0 END) AS null_record_source

FROM silver_jobtech.job_postings_silver_dedup;

## What the profiling tells us

The Silver table has **6,753,458 rows**. Some fields are very strong, some are partially complete, and some are too sparse to trust directly.

---

## Strong columns

These look reliable enough to treat as **core Silver fields**:

- `job_id` → 0 nulls  
- `job_title` → 0 nulls  
- `publication_date` → 0 nulls  
- `last_publication_date` → 0 nulls  
- `record_source` → 0 nulls  
- `employer_name` → only 66 nulls  
- `number_of_vacancies` → only 846 nulls  
- `occupation_label` → 8,382 nulls  
- `occupation_field_label` / `occupation_group_label` → 11,979 nulls each  

These missing counts are **tiny relative to 6.7M rows**

---

## Medium-quality columns

These are usable, but with caution:

- `municipality_code` → 140,846 nulls  
- `duration_label` → 227,842 nulls  
- `working_hours_type_label` → 227,824 nulls  
- `application_deadline` → 715,291 nulls  

These can still be useful in Silver and possibly Gold, but we should expect **unknown values**

---

## Weak / sparse columns

These are much less reliable:

- `city` → 3,165,861 nulls  
- `employment_type_label` → 1,406,889 nulls  
- `salary_type_label` → 1,402,353 nulls  
- `scope_of_work_min` → 3,097,620 nulls  
- `scope_of_work_max` → 3,135,768 nulls  
- `webpage_url` → 6,374,847 nulls  
- `removed_date` → 6,753,458 nulls  

`removed_date` is completely null → **not useful right now**

---

## Practical interpretation

### Keep as core fields

- `job_id`  
- `job_title`  
- `employer_name`  
- `occupation_label`  
- `occupation_field_label`  
- `occupation_group_label`  
- `municipality_code`  
- `city`  
- `employment_type_label`  
- `duration_label`  
- `working_hours_type_label`  
- `salary_type_label`  
- `number_of_vacancies`  
- `application_deadline`  
- `publication_date`  
- `last_publication_date`  
- `removed`  
- `record_source`  

---

### Keep, but clean / standardize

- Text label fields  
- Date fields  
- Vacancy field  
- Scope fields  

---

### Drop from polished Silver

- `removed_date` → entirely null  
- `webpage_url` → optional (keep only if needed for traceability)  
- `description` (struct) → skip for now (will be used later for RAG)

In [0]:
%sql
SELECT
    SUM(CASE WHEN TRIM(job_title) = '' THEN 1 ELSE 0 END) AS blank_job_title,
    SUM(CASE WHEN employer_name IS NOT NULL AND TRIM(employer_name) = '' THEN 1 ELSE 0 END) AS blank_employer_name,
    SUM(CASE WHEN occupation_label IS NOT NULL AND TRIM(occupation_label) = '' THEN 1 ELSE 0 END) AS blank_occupation_label,
    SUM(CASE WHEN occupation_field_label IS NOT NULL AND TRIM(occupation_field_label) = '' THEN 1 ELSE 0 END) AS blank_occupation_field_label,
    SUM(CASE WHEN occupation_group_label IS NOT NULL AND TRIM(occupation_group_label) = '' THEN 1 ELSE 0 END) AS blank_occupation_group_label,
    SUM(CASE WHEN city IS NOT NULL AND TRIM(city) = '' THEN 1 ELSE 0 END) AS blank_city,
    SUM(CASE WHEN employment_type_label IS NOT NULL AND TRIM(employment_type_label) = '' THEN 1 ELSE 0 END) AS blank_employment_type_label,
    SUM(CASE WHEN duration_label IS NOT NULL AND TRIM(duration_label) = '' THEN 1 ELSE 0 END) AS blank_duration_label,
    SUM(CASE WHEN working_hours_type_label IS NOT NULL AND TRIM(working_hours_type_label) = '' THEN 1 ELSE 0 END) AS blank_working_hours_type_label,
    SUM(CASE WHEN salary_type_label IS NOT NULL AND TRIM(salary_type_label) = '' THEN 1 ELSE 0 END) AS blank_salary_type_label
FROM silver_jobtech.job_postings_silver_dedup;

## Interpretation of blank values

---

## Very strong (no issues at all)

These are fully clean (no blanks):

- `job_title`
- `employer_name`
- `occupation_label`
- `occupation_field_label`
- `occupation_group_label`
- `employment_type_label`
- `salary_type_label`

These are **Gold-ready dimension candidates**

---

## Minor issues (acceptable)

- `duration_label` → 20,333 blanks  
- `working_hours_type_label` → 20,315 blanks  

Out of **6.7M rows → negligible**

We can:
- keep them  
- optionally replace blanks with `"Unknown"`

---

## Notable issue

### `city`

- 550,311 blank values  
- plus **3.1M nulls earlier**

This confirms:

- `city` is **not reliable as a primary location field**

BUT:

- we still have `municipality_code` → much better

---

## Key conclusion

Use:

- `municipality_code` → **primary geographic key**  
- `city` → **optional descriptive attribute (low trust)**

In [0]:
%sql
SELECT
    MIN(publication_date) AS min_publication_date,
    MAX(publication_date) AS max_publication_date,
    MIN(last_publication_date) AS min_last_publication_date,
    MAX(last_publication_date) AS max_last_publication_date
FROM silver_jobtech.job_postings_silver_dedup;

## Interpretation of date results

---

## Problem

We have:

- `max_publication_date = 2099-01-01`
- `max_last_publication_date = 2099-01-02`

👉 These are **not real job posting dates**

Common in APIs:

- `2099` = **"no expiration" / "open-ended job ad"**

---

## Decision

We will:

- Keep real dates  
- Convert fake future dates to `NULL`

In [0]:
%sql
CREATE OR REPLACE TABLE silver_jobtech.job_postings_silver_clean AS
SELECT
    job_id,
    job_title,
    employer_name,

    occupation_label,
    occupation_field_label,
    occupation_group_label,

    municipality_code,
    city,

    employment_type_label,
    duration_label,
    working_hours_type_label,
    salary_type_label,

    number_of_vacancies,

    -- Clean dates (remove fake future dates)
    CASE 
        WHEN publication_date >= '2090-01-01' THEN NULL
        ELSE TO_TIMESTAMP(publication_date)
    END AS publication_date,

    CASE 
        WHEN last_publication_date >= '2090-01-01' THEN NULL
        ELSE TO_TIMESTAMP(last_publication_date)
    END AS last_publication_date,

    CASE 
        WHEN application_deadline >= '2090-01-01' THEN NULL
        ELSE TO_TIMESTAMP(application_deadline)
    END AS application_deadline,

    -- Standardize blanks to NULL
    NULLIF(TRIM(duration_label), '') AS duration_label_clean,
    NULLIF(TRIM(working_hours_type_label), '') AS working_hours_type_label_clean,

    scope_of_work_min,
    scope_of_work_max,

    removed,
    record_source

FROM silver_jobtech.job_postings_silver_dedup;